(course-alzheimer-trajectory-management)=

# Path A - Module 47: Trajectory Management (Slicing)

Real simulations produce gigabytes of data spread across multiple files. Managing these structures efficiently without losing track of time is a developer's challenge. 

In this module, you will learn to slice, dice, and concatenate your Alzheimer's trajectories using MolSysMT.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Let's use our Villin trajectory as a proxy for a long simulation
villin_traj = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.dcd']
villin_topo = systems['chicken villin HP35']['traj_chicken_villin_HP35_solvated.h5msm']
molsys = [villin_topo, villin_traj]

### 1. Structure Slicing
You don't always need every structure. You can use the `structure_indices` argument in almost every function to focus on a specific time window.

In [2]:
# Get the number of structures
n_structures = msm.get(molsys, element='system', n_structures=True)
print(f"Total structures: {n_structures}")

# Extract only the first 10 structures and the last structure
indices = list(range(10)) + [n_structures - 1]
short_traj = msm.extract(molsys, structure_indices=indices)

print(f"Structures in short_traj: {msm.get(short_traj, element='system', n_structures=True)}")

Total structures: 20


Structures in short_traj: 11


### 2. Concatenating Structures
If you have two separate simulation runs, you can join them into a single virtual object using `concatenate_structures()`. Coordinate-only trajectory parts do not need to duplicate the topology, but they must use the same atom count and ordering.

In [3]:
# Split our short_traj into two parts
part1 = msm.extract(short_traj, structure_indices=[0, 1, 2])
part2 = msm.extract(short_traj, structure_indices=[3, 4, 5])

# Concatenate them back
merged_traj = msm.concatenate_structures([part1, part2])

print(f"Structures in merged_traj: {msm.get(merged_traj, element='system', n_structures=True)}")

Structures in merged_traj: 6


### 3. Sub-sampling (Striding)
A common task is to take every Nth structure to reduce the data size for visualization or quick analysis.

In [4]:
# Take every 10th structure of the original system
sub_sampled = msm.extract(molsys, structure_indices=range(0, n_structures, 10))

print(f"Original: {n_structures} structures -> Sub-sampled: {msm.get(sub_sampled, element='system', n_structures=True)} structures")

Original: 20 structures -> Sub-sampled: 2 structures


--- 

Tier-1 XYZ, DCD, and XTC conversion routes have exhaustive coordinate-trajectory reports. Atom selections are materialized in canonical increasing order, while `structure_indices` keeps the requested order; target formats that cannot carry a value report the omission explicitly.

### 🏆 Path A Challenge: The Data Curator

1. Create a trajectory that contains only the **even structures** (0, 2, 4...) of the Villin simulation.
2. Append the **last 5 structures** of the simulation to the end of your new trajectory.
3. Use `msm.get(..., structure_id=True)` to see if the structure IDs are preserved or if they are reindexed.

Managing data is the boring but necessary part of pipeline development. In **Module 48**, we will see the "Heavy Artillery": how to process files that are **larger than your RAM** using Iterators and the ChunkedExecutor.